In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Data Preparation

#### Load the dataset

In [ ]:
data = pd.read_csv(r"C:\Users\ntjam\OneDrive\Github\SML-Predicting Employee Attrition\employee_attrition.csv")

In [ ]:
# Display the first few rows

data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

### Remove irrelevant features

In [ ]:
data.drop("EmployeeID", axis=1, inplace=True) # employee id does not affect the fact that an employee will leave

In [ ]:
data.head()

#### Handle Missing Values
In this step, we check for missing values and decide on an appropriate imputation or removal strategy.

In [ ]:
data.isnull().sum()

In [ ]:
# In percentage
data.isnull().mean()*100

#### Lets visualize the missing values for easy communications with HR/Stakeholders

In [ ]:
import missingno as msno

msno.bar(data, color='sandybrown')

In [ ]:
# Display columns with missing values
sns.heatmap(data.isnull())

#### Imputation Strategies
**Numerical Columns:**
- Use `mean`, `median`, or `mode` to impute.
- For features like `Age`, median is often more robust against outliers.

**Categorical Columns:**
- Impute with the mode or create a new category (e.g., "Unknown").

**Advanced Options:**
- Use K-Nearest Neighbors (KNN) Imputation or regression models for more accurate imputation.

In [ ]:
# Impute missing numerical values with median
data['Age'] = data['Age'].fillna(data['Age'].median())
data['MonthlyIncome'] = data['MonthlyIncome'].fillna(data['MonthlyIncome'].median())

In [ ]:
# Impute missing categorical values with mode
data['JobSatisfaction'] = data['JobSatisfaction'].fillna(data['JobSatisfaction'].mode()[0])

In [ ]:
data.isnull().sum()

### Handling Duplicate Data
Detect Duplicates

In [ ]:
# Check for duplicate rows
data.duplicated().sum()

In [ ]:
len(data)

### Remove Duplicates
- Duplicates can often be dropped unless there’s a need to retain them for specific analysis.

In [ ]:
# Remove duplicate rows
data = data.drop_duplicates().reset_index(drop=True)

In [ ]:
len(data)

### Handle Outliers
Check for outliers in numerical columns.<br>
Lets investigate the numerical columns for outliers by visualizing their distributions on histplot

In [ ]:
numerical_columns = data.select_dtypes(include="number")
fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(20, 20))
ax=ax.flatten()
for idx, col in enumerate(numerical_columns):
    sns.histplot(data[col], ax=ax[idx])
    ax[idx].set_title(f"Histplot for {col}")
plt.show()

#### We can use Boxplot to further confirm the outliers in numerical columns

In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(20, 20))
ax=ax.flatten()
for idx, col in enumerate(numerical_columns):
    sns.boxplot(data[col], ax=ax[idx])
    ax[idx].set_title(f"Boxplot for {col}")
plt.show()

### Lets remove the outliers on affected columns using IQR (InterQuaterRange)

In [ ]:
affected_column = ["Age", "MonthlyIncome", "YearsAtCompany", "TrainingTimesLastYear"]

In [ ]:
q1 = data[affected_column].quantile(0.25)
q3 = data[affected_column].quantile(0.75)

In [ ]:
iqr = q3 - q1

In [ ]:
lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

In [ ]:
# clip value into lower bound and upper bound. Value will not exceed lower and upper bounds

data[affected_column] = data[affected_column].clip(lower=lower_bound, upper=upper_bound, axis=1) 

### Lets verify that our data is now outlier free

In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(20, 20))
ax=ax.flatten()
for idx, col in enumerate(numerical_columns):
    sns.boxplot(data[col], ax=ax[idx])
    ax[idx].set_title(f"Boxplot for {col}")
plt.show()

### Encode Categorical Variables
Convert categorical variables like `Gender`, `Department`, and `OverTime` into numeric formats.

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Label encoding for binary categories

label_enc = LabelEncoder()
data['Gender'] = label_enc.fit_transform(data['Gender'])  # Fit and transform
data['OverTime'] = label_enc.fit_transform(data['OverTime'])
data['Attrition'] = label_enc.fit_transform(data['Attrition'])

In [ ]:
# # One-hot encoding for multi-class categorical variables

data = pd.get_dummies(data, columns=['Department'], drop_first = True) #dtype=int to replace drop_first = True

In [ ]:
# Display the transformed dataset
data.head()

### Scale Numerical Features
Scale numerical columns to bring them to a similar range, which can improve model performance.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
# Standard scaler = Normal or slightly skewed distribution 
# MinMax scaler = Uniform distribution
# Robust Scaler = Heavily distribution

scaler = StandardScaler() #because monthly, years, age are slightly skewed
num_cols_to_scale = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'TrainingTimesLastYear']
data[num_cols_to_scale] = scaler.fit_transform(data[num_cols_to_scale])

In [ ]:
# Verify scaling

data[num_cols_to_scale].describe()

### Check Class Distribution
Assess the balance of the target variable `(Attrition)` to determine if we need to handle class imbalance.

In [ ]:
# Check class distribution
data['Attrition'].value_counts()

In [ ]:
# Normalise the data

data['Attrition'].value_counts(normalize=True)*100

In [ ]:
sns.countplot(x=data['Attrition'])

### Handling Class Imbalance
##### Techniques to Address Class Imbalance
- Resampling:
  - Oversampling the minority class using techniques like SMOTE (Synthetic Minority Oversampling Technique).
  - Undersampling the majority class to balance proportions.
- Class Weights:
    - Add weights to the classes in the loss function during model training to penalize misclassifications of the minority class.
- Evaluation Metrics:
    - Use metrics like precision, recall, F1-score, or ROC-AUC instead of just accuracy.


#### Oversampling with SMOTE

In [ ]:
# pip install --upgrade imbalanced-learn

from imblearn.over_sampling import SMOTE

In [ ]:
# Separate features and target

X = data.drop('Attrition', axis=1)
y = data['Attrition']

In [ ]:
# Apply SMOTE to oversample the minority class

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
y_resampled.value_counts()

In [ ]:
y_resampled.value_counts(normalize=True)*100

In [ ]:
sns.countplot(x=y_resampled)

In [ ]:
resampled_data = pd.DataFrame(X_resampled, columns = X.columns)

In [ ]:
resampled_data.head()

In [ ]:
resampled_data["Attrition"] = y_resampled

In [ ]:
resampled_data

### Final Dataset Ready
At this stage, the dataset is clean and prepared for the next step in the workflow.

In [ ]:
# Save the cleaned dataset for future use

resampled_data.to_csv('SML2_cleaned_employee_attrition_balanced.csv', index=False)

In [ ]:
df = pd.read_csv("SML2_cleaned_employee_attrition_balanced.csv")

In [ ]:
df["Attrition"].value_counts()

### Data Splitting
We’ll split the cleaned dataset into training and testing sets, ensuring the target variable’s distribution is representative in both.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Split the dataset (80% training, 20% testing)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled  # Stratify to maintain class proportions
)

In [ ]:
print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

In [ ]:
# Proportion of Train set distribution

y_train.value_counts(normalize=True)*100

In [ ]:
# Proportion of Test set distribution

y_test.value_counts(normalize=True)*100

### Model Selection
We’ll train multiple classification models to find the best one for predicting attrition. Start with a baseline and expand to more advanced models.

### 1. Train a Baseline Model (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Train logistic regression model
logreg = LogisticRegression(random_state=42)

# logreg = LogisticRegression(class_weight='balanced', random_state=42)
logreg.fit(X_train, y_train)

In [ ]:
# Predict on the test set
y_pred = logreg.predict(X_test)

In [ ]:
# Evaluate performance
print("Logistic Regression Classification Report:\n")
print(classification_report(y_test, y_pred))

### 2. Train Advanced Models (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train random forest model
# rf = RandomForestClassifier(class_weight={0:1, 1:3}, random_state=42, n_estimators=100)
rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

# Predict on the test set
y_pred_rf = rf.predict(X_test)

# Evaluate performance
print("Random Forest Classification Report:\n")
print(classification_report(y_test, y_pred_rf))

### 3. Trying multiple classification models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
!pip install xgboost
from xgboost import XGBClassifier

In [ ]:
# Define the models
# class_weights={0:1, 1:3}
class_weights='balanced'
models = {
    'Logistic Regression': LogisticRegression(random_state=42, class_weight=class_weights),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight=class_weights),
    'Support Vector Machine': SVC(random_state=42, class_weight=class_weights),
    'k-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight=class_weights),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42, algorithm="SAMME"),
    'XGBoost': XGBClassifier(eval_metric='mlogloss')
}

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

results = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    matrix = confusion_matrix(y_test, y_pred)
    
    # Store results
    results[model_name] = {
        'accuracy': accuracy,
        'classification_report': report,
        'confusion_matrix': matrix
    }
    # print(f"{model_name} Accuracy: {accuracy:.4f}")
    print(f"{model_name} Classification Report:\n{report}")

    sns.heatmap(matrix, annot=True, fmt="g")
    plt.show()

### 4. Hyperparameter Tuning
We’ll use grid search to optimize hyperparameters for the Random Forest model.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [ ]:
grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_grid, cv=5, scoring='f1', n_jobs=-1
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters
print("Best Hyperparameters:", grid_search.best_params_)

In [ ]:
# Train best model
best_rf = grid_search.best_estimator_

In [ ]:
# Predict on the test set
y_pred_best_rf = best_rf.predict(X_test)

In [ ]:
# Evaluate performance
print("Tuned Random Forest Classification Report:\n")
print(classification_report(y_test, y_pred_best_rf))

### Model Evaluation
We’ll perform a detailed evaluation using metrics like precision, recall, F1-score, and AUC-ROC. Confusion matrices and AUC curves will help visualize performance.

#### Confusion Matrix
Confusion matrix to visualize misclassifications.

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred_best_rf)

In [ ]:
# Plot the confusion matrix
plt.figure(figsize=(8, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


#### ROC Curve and AUC Score
ROC-AUC to evaluate the model's ability to distinguish between classes.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
# Compute probabilities for ROC
y_probs = best_rf.predict_proba(X_test)[:, 1]  # Probabilities for the positive class

In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_probs) #fpr: false positive rate, tpr: true positive rate
auc_score = roc_auc_score(y_test, y_probs)

In [ ]:
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.show()

### 6. Using AutoML to select best model

#### Using PyCaret
PyCaret is a low-code library for automating machine learning workflows.

In [168]:
#pip install pycaret

  Using cached pycaret-3.3.2-py3-none-any.whl.metadata (17 kB)
  Using cached numpy-1.26.4-cp313-cp313-win_amd64.whl
  Using cached pandas-2.1.4.tar.gz (4.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [32 lines of output]
  + meson setup C:\Users\ntjam\AppData\Local\Temp\pip-install-9qph4yzy\pandas_64e1cd3b62c4439a953fd796ee01963a C:\Users\ntjam\AppData\Local\Temp\pip-install-9qph4yzy\pandas_64e1cd3b62c4439a953fd796ee01963a\.mesonpy-c2bl0ohj\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\ntjam\AppData\Local\Temp\pip-install-9qph4yzy\pandas_64e1cd3b62c4439a953fd796ee01963a\.mesonpy-c2bl0ohj\build\meson-python-native-file.ini
  The Meson build system
  Version: 1.2.1
  Source dir: C:\Users\ntjam\AppData\Local\Temp\pip-install-9qph4yzy\pandas_64e1cd3b62c4439a953fd796ee01963a
  Build dir: C:\Users\ntjam\AppData\Local\Temp\pip-install-9qph4yzy\pandas_64e1cd3b62c4439a953fd796ee01963a\.mesonpy-c2bl0ohj\build
  Build type: native build
  Project name: pandas
  Project version: 2.1.4
  Activating VS 18.7.2
  C compiler for t

In [167]:
#pip install scikit-learn==1.4

     ---------------------------------------- 0.0/7.7 MB ? eta -:--:--
     ------------ --------------------------- 2.4/7.7 MB 13.4 MB/s eta 0:00:01
     ----------------------------- ---------- 5.8/7.7 MB 14.6 MB/s eta 0:00:01
     ---------------------------------------- 7.7/7.7 MB 15.0 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): still running...
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for scikit-learn: filename=scikit_learn-1.4.0-cp313-cp313-win_amd64.whl size=9337677 sha256=af26237a551a3ccf36abae634b54841b4bd6e2ec269cd369c14e6f5726c3abac
  Stored in directory: c:\users\ntjam\appdata\local\pip\cache\wheels\47\53\1d\225f16bc35c12a3f8132ae16be077e61ea49e63da03ae0c674
Success

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imbalanced-learn 0.14.0 requires scikit-learn<2,>=1.4.2, but you have scikit-learn 1.4.0 which is incompatible.


In [169]:
from pycaret.classification import *

ModuleNotFoundError: No module named 'pycaret'

In [ ]:
# Combine features and target into one DataFrame
# df = pd.concat([X, y], axis=1)
df = pd.concat([X_resampled, y_resampled], axis=1)

In [ ]:
df.head(2)

In [ ]:
# Initialize PyCaret
clf_setup = setup(
    data=df, 
    target='Attrition', 
    normalize=True,
    session_id=42,
    # use_gpu=True,
)

In [ ]:
# Compare models and select the best one
best_model = compare_models(cross_validation=False, sort="f1", )

In [ ]:
# Display the best model
print(best_model)

In [ ]:
# Fine-tune the best model
tuned_model = tune_model(best_model, optimize="f1", fold=5)

In [ ]:
# Evaluate the tuned model
evaluate_model(tuned_model, fold=5)

In [ ]:
# Finalize the model
final_model = finalize_model(tuned_model)

In [ ]:
# Predict on the test set
predictions = predict_model(final_model, data=X_test)

#### Saving the model

In [ ]:
save_model(final_model, 'final_model_attrition_prediction')

#### Loading the model

In [ ]:
loaded_model = load_model('final_model_attrition_prediction')
